# Notebook 03 — Treinamento dos Modelos

**Objetivo:** Vetorizar os textos pré-processados e treinar os 4 algoritmos de classificação com 2 estratégias de vetorização, totalizando **8 experimentos**.

## Estratégia experimental

| Algoritmo | Justificativa |
|---|---|
| **Naive Bayes Multinomial** | Baseline clássico para classificação de texto; assume independência entre features |
| **Regressão Logística** | Modelo linear eficiente; costuma ser o melhor com TF-IDF |
| **SVM (LinearSVC)** | Robusto em espaços de alta dimensão (vocabulário vetorizado) |
| **Random Forest** | Representa modelos baseados em árvores; mostra que complexidade nem sempre = melhor resultado |

| Vetorização | Descrição |
|---|---|
| **Bag of Words (BoW)** | Conta a frequência de cada palavra; trata todas com o mesmo peso |
| **TF-IDF** | Pondera palavras pela frequência no documento e raridade no corpus; penaliza termos muito comuns |

**Total:** 4 modelos × 2 vetorizações = **8 experimentos**

In [26]:
# ===== SETUP: montar o Drive =====
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/tcc-sentiment-analysis/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Importações

In [27]:
import pandas as pd
import joblib # para salvar e carregar modelos treinados (.pkl)
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

## 2. Carregamento dos Dados e Divisão Treino/Teste

Carregamos o dataset pré-processado gerado pelo notebook anterior.

**Divisão 80/20 com `stratify=y`:**
- **80%** dos dados para treino (40.000 reviews)
- **20%** para teste (10.000 reviews)
- `stratify=y` garante que a proporção de classes (50% positivo / 50% negativo) seja mantida em ambos os conjuntos
- `random_state=42` garante reprodutibilidade — o mesmo split é gerado sempre que o código for executado

In [28]:
# Carregar o dataset processado gerado pelo notebook 02
df = pd.read_csv(DRIVE_PATH + 'data/imdb_clean.csv')

# Converter o rótulo textual em binário: positive=1, negative=0
df['label'] = (df['sentiment'] == 'positive').astype(int)

X = df['review_clean']  # features: texto pré-processado
y = df['label']          # target: 0 (negativo) ou 1 (positivo)

# Divisão treino/teste estratificada e reprodutível
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para teste
    random_state=42,     # semente para reprodutibilidade
    stratify=y           # mantém proporção de classes
)

print(f"Treino: {len(X_train)} reviews | Teste: {len(X_test)} reviews")

Treino: 40000 reviews | Teste: 10000 reviews


## 3. Definição dos Modelos e Vetorizadores

Definimos os 4 modelos e as 2 estratégias de vetorização que serão combinados nos experimentos.

**Parâmetro `max_features=10000`:** limita o vocabulário às 10.000 palavras mais frequentes, reduzindo dimensionalidade e tempo de treinamento sem perda significativa de informação.

In [29]:
# Vetorizadores: convertem texto em matriz numérica
vetorizadores = {
    'BoW': CountVectorizer(max_features=10000),    # Bag of Words: frequência absoluta
    'TF-IDF': TfidfVectorizer(max_features=10000)  # TF-IDF: frequência ponderada por raridade
}

# Modelos de classificação
modelos = {
    'Naive_Bayes': MultinomialNB(),                                        # baseline probabilístico
    'Regressao_Logistica': LogisticRegression(max_iter=1000),              # modelo linear
    'SVM': LinearSVC(max_iter=1000),                                       # máquina de vetores de suporte linear
    'Random_Forest': RandomForestClassifier(n_estimators=100, random_state=42)  # ensemble de árvores
}

print("✅ Modelos e vetorizadores definidos")

✅ Modelos e vetorizadores definidos


## 4. Treinamento dos 8 Experimentos

Para cada combinação (vetorizador × modelo):
1. O vetorizador é ajustado no conjunto de **treino** (`fit_transform`) e apenas aplicado no **teste** (`transform`) — evitando *data leakage*
2. O modelo é treinado e avaliado
3. Tanto o vetorizador quanto o modelo são salvos como `.pkl` para uso no Streamlit

>  **Por que salvar o vetorizador?** O vetorizador treinado no conjunto de treino "conhece" o vocabulário. Ao fazer predições em produção (no Streamlit), o mesmo vetorizador deve ser usado para transformar o novo texto — caso contrário, as features serão incompatíveis com o modelo.

>  **Tempo estimado:** ~7 minutos (Random Forest é o mais demorado)

In [30]:
# Criar pasta para salvar os modelos, se não existir
MODELS_PATH = DRIVE_PATH + 'models/'
os.makedirs(MODELS_PATH, exist_ok=True)

resultados = []  # lista que acumulará as métricas de cada experimento

for vet_nome, vet in vetorizadores.items():
    # fit_transform: aprende o vocabulário E transforma o treino
    X_train_vec = vet.fit_transform(X_train)
    # transform: apenas aplica o vocabulário já aprendido ao teste
    X_test_vec = vet.transform(X_test)

    # Salvar o vetorizador treinado — necessário para o Streamlit
    joblib.dump(vet, MODELS_PATH + f'vectorizer_{vet_nome}.pkl')
    print(f"💾 Vetorizador salvo: vectorizer_{vet_nome}.pkl")

    for mod_nome, mod in modelos.items():
        print(f"⏳ Treinando {mod_nome} + {vet_nome}...")

        # Treinar o modelo com os dados vetorizados
        mod.fit(X_train_vec, y_train)

        # Gerar predições no conjunto de teste
        y_pred = mod.predict(X_test_vec)

        # Salvar o modelo treinado — necessário para o Streamlit
        joblib.dump(mod, MODELS_PATH + f'{mod_nome}_{vet_nome}.pkl')

        # Calcular e registrar métricas de avaliação
        resultados.append({
            'Modelo': mod_nome,
            'Vetorização': vet_nome,
            'Acurácia': round(accuracy_score(y_test, y_pred), 4),
            'Precisão': round(precision_score(y_test, y_pred), 4),
            'Recall': round(recall_score(y_test, y_pred), 4),
            'F1-Score': round(f1_score(y_test, y_pred), 4)  # métrica principal
        })
        print(f"✅ {mod_nome} + {vet_nome} — F1: {resultados[-1]['F1-Score']}")

print("\n🎉 Todos os modelos treinados e salvos!")

💾 Vetorizador salvo: vectorizer_BoW.pkl
⏳ Treinando Naive_Bayes + BoW...
✅ Naive_Bayes + BoW — F1: 0.8406
⏳ Treinando Regressao_Logistica + BoW...
✅ Regressao_Logistica + BoW — F1: 0.8713
⏳ Treinando SVM + BoW...
✅ SVM + BoW — F1: 0.8434
⏳ Treinando Random_Forest + BoW...
✅ Random_Forest + BoW — F1: 0.847
💾 Vetorizador salvo: vectorizer_TF-IDF.pkl
⏳ Treinando Naive_Bayes + TF-IDF...
✅ Naive_Bayes + TF-IDF — F1: 0.8518
⏳ Treinando Regressao_Logistica + TF-IDF...
✅ Regressao_Logistica + TF-IDF — F1: 0.8893
⏳ Treinando SVM + TF-IDF...
✅ SVM + TF-IDF — F1: 0.8834
⏳ Treinando Random_Forest + TF-IDF...
✅ Random_Forest + TF-IDF — F1: 0.8467

🎉 Todos os modelos treinados e salvos!


## 5. Tabela Comparativa dos 8 Experimentos

Exibimos e salvamos a tabela com todas as métricas ordenadas por **F1-Score** (métrica principal).

**Por que F1-Score como métrica principal?**  
O F1 é a média harmônica entre Precisão e Recall, equilibrando os dois aspectos da classificação. Em um dataset balanceado como o IMDB, o F1 e a Acurácia tendem a ser próximos, mas o F1 é mais robusto e amplamente usado na literatura de NLP.

In [31]:
# Construir e exibir a tabela comparativa dos 8 experimentos
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values('F1-Score', ascending=False)  # melhor primeiro

print(df_resultados.to_string(index=False))

# Salvar a tabela para uso no notebook 04 e no TCC
df_resultados.to_csv(DRIVE_PATH + 'results/resultados.csv', index=False)
print("\n✅ resultados.csv salvo em: results/resultados.csv")

             Modelo Vetorização  Acurácia  Precisão  Recall  F1-Score
Regressao_Logistica      TF-IDF    0.8883    0.8812  0.8976    0.8893
                SVM      TF-IDF    0.8826    0.8772  0.8898    0.8834
Regressao_Logistica         BoW    0.8709    0.8688  0.8738    0.8713
        Naive_Bayes      TF-IDF    0.8513    0.8490  0.8546    0.8518
      Random_Forest         BoW    0.8475    0.8499  0.8440    0.8470
      Random_Forest      TF-IDF    0.8487    0.8581  0.8356    0.8467
                SVM         BoW    0.8437    0.8450  0.8418    0.8434
        Naive_Bayes         BoW    0.8426    0.8513  0.8302    0.8406

✅ resultados.csv salvo em: results/resultados.csv


---
## Conclusões do Treinamento

- **8 experimentos** realizados com sucesso (4 modelos × 2 vetorizações)
- Todos os modelos e vetorizadores salvos como `.pkl` na pasta `models/`
- Tabela de resultados salva como `results/resultados.csv`
- **TF-IDF superou BoW** em todos os 4 modelos
- **Regressão Logística + TF-IDF** apresentou o maior F1-Score

**Próximo passo:** `04_results.ipynb` — visualização e análise dos resultados